In [ ]:
# =========================================================
# 1. IMPORTS
# =========================================================
print("Loading imports...")
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore') # Suppresses Pandas/Sklearn deprecation warnings for cleaner output

from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import GridSearchCV
from skopt import BayesSearchCV
from skopt.space import Real
from MultiPiecwiseRegressor import *
from Last15DaysSplit import *

# =========================================================
# 2. LOAD DATA
# =========================================================
print("Loading data...")
# parse_dates ensures date columns are loaded as datetime objects, saving a conversion step later
train = pd.read_csv('store-sales-time-series-forecasting/train.csv', parse_dates=['date'])
test = pd.read_csv('store-sales-time-series-forecasting/test.csv', parse_dates=['date'])
holidays_events = pd.read_csv('store-sales-time-series-forecasting/holidays_events.csv', parse_dates=['date'])

# =========================================================
# 3. DATA MERGING & PREPARATION
# =========================================================
print("Merging data...")
# Prepare Holidays
# In Ecuador, if a holiday falls on a weekend, it is often 'transferred' to a weekday.
# We filter out the original date of transferred holidays so we don't count them twice.
valid_holidays = holidays_events[holidays_events['transferred'] == False].drop_duplicates(subset=['date'])
valid_holidays = valid_holidays[['date', 'type']].rename(columns={'type': 'holiday_type'})

train = train.merge(valid_holidays, on='date', how='left')
test = test.merge(valid_holidays, on='date', how='left')

# Combine train and test to ensure consistent feature engineering
# This prevents errors where a specific category (e.g., a specific holiday) appears 
# in the test set but not the training set, which would break the OneHotEncoder.
train['is_test'] = False
test['is_test'] = True
test['sales'] = 0.0 # Placeholder target for the test set

df = pd.concat([train, test], ignore_index=True)

# The competition metric is Root Mean Squared Logarithmic Error (RMSLE).
# By applying log1p (log(1 + x)) to the target now, we can just use standard 
# RMSE as our loss function in the model.
df['log1p_sales'] = np.log1p(df['sales'])

# =========================================================
# 4. FEATURE ENGINEERING
# =========================================================
print("Engineering features...")

# --- Date & Fourier Features ---
# Day of week is strictly categorical
df['dayofweek'] = df['date'].dt.dayofweek.astype(str)
df['dayofyear'] = df['date'].dt.dayofyear

# Fourier terms (sin/cos) model annual seasonality effectively. 
for i in range(1, 4):
    df[f'sin_{i}'] = np.sin(2 * np.pi * i * df['dayofyear'] / 365.25)
    df[f'cos_{i}'] = np.cos(2 * np.pi * i * df['dayofyear'] / 365.25)

# --- Earthquake Feature ---
# The devastating earthquake occurred on April 16, 2016. It drastically affected 
# Affected supermarket sales (panic buying, relief supplies) for ~4 weeks.
df['earthquake_impact'] = ((df['date'] >= '2016-04-16') & (df['date'] <= '2016-05-16')).astype(int)

# --- Payday Feature ---
# Wages in the public sector are paid every 15th and on the last day of the month.
# Sales spike during these times. This calculates days since the most recent payday.
df['day'] = df['date'].dt.day
df['is_month_end'] = df['date'].dt.is_month_end
df['days_since_payday'] = np.where(
    df['day'] < 15, 
    df['day'],  # Logic for the 1st through the 14th
    np.where(
        df['day'] == 15,
        0,          # The 15th is payday
        np.where(
            df['is_month_end'],
            0,          # The last day of the month is payday
            df['day'] - 15  # Logic for the 16th up to the day before month-end
        )
    )
)
df = df.drop(columns=['day', 'is_month_end'])

# --- Holiday Features ---
# Convert holiday presence into a simple binary feature
df['is_holiday'] = df['holiday_type'].notnull().astype(int)

# --- Global Lag Features (16, 21, and 28 Days) ---
# Group by store and family, then shift the target variable backwards.
# The Kaggle test set is 15 days long, so we use 16+ day lags to avoid 
# the need for recursive predictions.
# 16 days is used as it is the soonest available.
# 21 and 28 days represent exactly 3 and 4 weeks prior, aligning with day-of-week shopping habits.
lag_days = [16, 21, 28]
for lag in lag_days:
    df[f'lag_{lag}'] = df.groupby(['store_nbr', 'family'])['log1p_sales'].shift(lag)

# Shifting creates NaNs at the start of the timeline. We fill with 0 temporarily.
# We will drop the first 28 days of the training set later to prevent training on this artificial data.
lag_cols = [f'lag_{lag}' for lag in lag_days]
df[lag_cols] = df[lag_cols].fillna(0)

# --- Categorical Encoding ---
# One-hot encode categoricals globally so train and test perfectly align.
# We do NOT encode 'store_nbr' and 'family' here because the router needs them intact.
# We do not set drop_first=True since we will use Ridge which will handle the multi-collinearity.
df = pd.get_dummies(df, columns=['dayofweek'])

# =========================================================
# 5. SETUP ROUTING
# =========================================================
print("Setting up routing logic...")

def store_family_router(X):
    """
    Creates routing keys in the format "{store_nbr}_{family}".
    Assumes X is a pandas DataFrame containing these columns.
    """
    # Cast store_nbr to int (in case it was read as float) then to string
    store_str = X['store_nbr'].astype(int).astype(str)
    
    # Cast family to string
    family_str = X['family'].astype(str)
    
    # Concatenate them with an underscore
    routes = store_str + "_" + family_str
    
    # Return as a numpy array so the custom estimator's masking logic works perfectly
    return routes.to_numpy()

# Use the routing function to get an array of all keys in the training data
all_training_keys = store_family_router(train)

# Extract just the unique keys
unique_keys = np.unique(all_training_keys)

# Define columns that shouldn't be passed to the models as numerical features
cols_to_drop = [
    'id', 'date', 'store_nbr', 'family', 'sales', 'log1p_sales', 
    'is_test', 'holiday_type', 'dayofyear'
]

# =========================================================
# 6. TRAIN / VALIDATION SPLIT
# =========================================================
print("Splitting data...")
train_full = df[~df['is_test']].copy()
test_full = df[df['is_test']].copy()

# Filter out the first 28 days of training data since our lag_28 feature is entirely 
# NaNs (filled with 0s) during this period.
train_full = train_full[train_full['date'] >= train_full['date'].min() + pd.Timedelta(days=28)]

# Standard temporal split: use the last 15 days of August 2017 as validation, 
# mimicking the public test set timeframe.
val_start_date = '2017-08-01'

X_train = train_full[train_full['date'] < val_start_date].copy()
y_train = X_train['log1p_sales']

X_val = train_full[train_full['date'] >= val_start_date].copy()
y_val = X_val['log1p_sales']

# Combine them for the SearchCV
X_search = pd.concat([X_train, X_val])
y_search = pd.concat([y_train, y_val])

# =========================================================
# 7. SETUP ESTIMATORS
# =========================================================
print("Setting up base estimators...")
# set up search space for Out-Of_Sample Validation
search_space = {
    # Search for the optimal L2 regularization strength (alpha) on a logarithmic scale
    'alpha': Real(1e-3, 1e1, prior='log-uniform')
}

# set up Bayes Search Template to copy into my_estimators_OOS dictionary for Out-Of_Sample Validation
bayes_search_template = BayesSearchCV(
    estimator=Ridge(),
    search_spaces=search_space,
    n_iter=15,                              
    cv=Last15DaysSplit(),   # Isolate the last 15 days to test on, train on the rest.                             
    scoring='neg_root_mean_squared_error',  
    n_jobs=1,                          
    random_state=0,
    verbose=0
)

# Create the OOS dictionary dynamically
my_estimators_OOS = {key: clone(bayes_search_template) for key in unique_keys}

# set up parameter grid for Cross Validation
param_grid = {'alpha': [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000, 100000, 1000000]}

# set up Grid Search Template to copy into my_estimators_CV dictionary for Cross Validation
grid_search_template = GridSearchCV(
    estimator=Ridge(),
    param_grid=param_grid,                             
    cv=TimeSeriesSplit(n_splits=5),                             
    scoring='neg_root_mean_squared_error',  
    n_jobs=1,                          
    verbose=0
)

# Create the CV dictionary dynamically
my_estimators_CV = {key: clone(grid_search_template) for key in unique_keys}

# =========================================================
# 8. VALIDATION EVALUATION
# =========================================================
print("\nTraining on pre-August data for validation using OOS...")

val_model_OOS = MultiPiecewiseRegressor(
    my_estimators_OOS, 
    store_family_router, 
    verbose=0, 
    n_jobs=-1, 
    drop_cols=cols_to_drop
)

val_model_OOS.fit(X_search, y_search)

print("\nTraining on pre-August data for validation using CV...")

val_model_CV = MultiPiecewiseRegressor(
    my_estimators_CV, 
    store_family_router, 
    verbose=0, 
    n_jobs=-1, 
    drop_cols=cols_to_drop
)

val_model_CV.fit(X_train, y_train)

print("Predicting on OOS validation set...")
val_preds_OOS = val_model_OOS.predict(X_val)

print("Predicting on CV validation set...")
val_preds_CV = val_model_CV.predict(X_val)

# ReLU clipping: Linear models can sometimes predict negative sales.
# We cap the minimum prediction at 0, as you cannot sell negative items.
val_preds_OOS_relu = np.maximum(0, val_preds_OOS)
val_preds_CV_relu = np.maximum(0, val_preds_CV)

val_rmsle_OOS = root_mean_squared_error(y_val, val_preds_OOS_relu)
print(f"Out-of-Sample RMSLE: {val_rmsle_OOS:.4f}")

val_rmsle_CV = root_mean_squared_error(y_val, val_preds_CV_relu)
print(f"CV Out-of-Sample RMSLE: {val_rmsle_CV:.4f}")

# =========================================================
# 9. FULL TRAINING & SUBMISSION
# =========================================================
print("\nExtracting best parameters and retraining on full dataset for submission...")

# Create a dictionary of the already-tuned Ridge models,
# val_model_OOS.estimators_ contains the results of the first BayesSearch
# and similarly for val_model_CV
tuned_OOS_estimators = val_model_OOS.estimators_.copy()
tuned_CV_estimators = val_model_CV.estimators_.copy()

# Initialize the final models using these already tuned estimators
final_OOS_model = MultiPiecewiseRegressor(
    tuned_OOS_estimators,          # Pass the dict of fitted/tuned Ridge models
    store_family_router, 
    verbose=0, 
    n_jobs=-1, 
    drop_cols=cols_to_drop
)

final_CV_model = MultiPiecewiseRegressor(
    tuned_CV_estimators,          # Pass the dict of fitted/tuned Ridge models
    store_family_router, 
    verbose=0, 
    n_jobs=-1, 
    drop_cols=cols_to_drop
)

# Refit on ALL available training data (including validation data)
# to give the final model the absolute most recent trends before predicting the test set.
final_OOS_model.fit(train_full, train_full['log1p_sales'])
final_CV_model.fit(train_full, train_full['log1p_sales'])

print("Predicting on test set...")
test_preds_OOS = final_OOS_model.predict(test_full)
test_preds_OOS_relu = np.maximum(0, test_preds_OOS)

test_preds_CV = final_CV_model.predict(test_full)
test_preds_CV_relu = np.maximum(0, test_preds_CV)

# Format and save submission
submission_3_OOS = test_full[['id']].copy()
submission_3_CV = test_full[['id']].copy()
# Reverse the log1p transformation using expm1 to get actual sales figures
submission_3_OOS['sales'] = np.expm1(test_preds_OOS_relu)
submission_3_CV['sales'] = np.expm1(test_preds_CV_relu)

submission_3_OOS.to_csv('submission_3_OOS.csv', index=False)
print("Successfully generated submission_3_OOS.csv")

submission_3_CV.to_csv('submission_3_CV.csv', index=False)
print("Successfully generated submission_3_CV.csv")